# FLUT: quarterly expectations review

Keep the current native-state evidence beside Flutter's reported US segment revenue reference. State amounts are **USD**; company amounts are **USD millions**. The reference is derived from management's August guidance. It is not a house forecast, a forecast derived from the state panel, or an approved model change.

This notebook reads the retained database and sources. Saving a dated snapshot is disabled until you set an explicit new destination below. New captures cannot recreate historical point-in-time evidence.


In [ ]:
database_file = "data/staging/refresh_20260912T201854Z/gaming_current.sqlite"
through_month = "2026-07"  # exact Q3 window; July alone is one of three months
as_of = None  # None uses the real current UTC time; historical cutoffs reject later captures
scenario_inputs = None  # Optional: explicitly supply all four full-US inputs and a rationale
export_snapshot = False
snapshot_destination = None  # An absolute NEW directory is required if export_snapshot is True
snapshot_to_evaluate = None  # Optional existing expectation.json; no snapshot is written by evaluation
actual_file = None  # Optional exact-quarter actual JSON with retained source and public-release clocks


In [ ]:
from contextlib import closing
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import sqlite3
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if (ROOT / "gaming" / "src").is_dir():
    ROOT = ROOT / "gaming"
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from variant_gaming.flut_scorecard import build_monthly_scorecard, build_quarterly_scorecard
from variant_gaming.flut_expectations import (
    load_company_reference, build_expectations_review, analyst_scenario,
    freeze_expectation, evaluate_expectation,
)

DB = ROOT / database_file
manifest = json.loads((DB.parent / "run_manifest.json").read_text())
validation = json.loads((DB.parent / "validation.json").read_text())
if not manifest.get("finished_at"):
    raise RuntimeError("Capture is unfinished; inspect its collection log.")
before = hashlib.sha256(DB.read_bytes()).hexdigest()
if before != validation["database_sha256"]:
    raise RuntimeError("Database differs from its validated capture.")
with closing(sqlite3.connect(DB.resolve().as_uri() + "?mode=ro", uri=True)) as connection:
    observations = pd.read_sql_query("SELECT * FROM gaming_results", connection)
reference = load_company_reference(ROOT / "config/flut_company_reference.json", project_root=ROOT)
quarter = reference["management_reference"]["period"]
cutoff = as_of or datetime.now(timezone.utc).isoformat()
monthly = build_monthly_scorecard(observations)
quarterly = build_quarterly_scorecard(monthly, quarter=quarter, through_month=through_month)
review = build_expectations_review(quarterly, reference, as_of=cutoff,
    data_capture_at=manifest["finished_at"], database_path=DB)
pd.set_option("display.max_colwidth", None)
print("Quarter:", quarter, "| Through:", through_month, "| Information cutoff:", cutoff)
print("Capture:", manifest["finished_at"], "| Status: unapproved research")


## Native-state evidence

Each row preserves its revenue definition, coverage, and comparison window. Missing comparisons remain visible. A growth rate, hold change, or share change is evidence to review; no national revenue adjustment is inferred. See notebook 95 for reconciliation and the full coverage audit.


In [ ]:
display(quarterly[["state_code", "vertical", "metric", "native_metric", "through_month",
    "matched_window_months", "expected_window_months", "quarter_complete", "status",
    "fd_amount", "prior_fd_amount", "fd_growth_pct", "fd_share_pct", "share_change_pp"]])
print("Evidence-to-company adjustment:", review["state_to_company_adjustment"])
print("Translation status:", review["translation_status"])


## Separate company reference

The US management midpoint is **$1,480m**: $7,400m full-year guidance × approximately 20% Q3 phasing. The displayed range applies the same phasing to the full-year endpoints; management did not give a separate numerical Q3 range. Its US adjusted EBITDA expectation is approximately breakeven, not an exact $0 reported result.

Product-level quarterly revenue is not inferred using Q2's product mix. The prior-year Q3 and latest Q2 reported figures below remain separate from expectations. Source PDFs and their exact hashes are visible.


In [ ]:
display(pd.DataFrame([reference["management_reference"]])[["period", "scope", "kind", "unit",
    "value_low", "value_mid", "value_high", "formula", "range_limitation"]])
display(pd.DataFrame(reference["reported_actuals"])[["period", "scope", "metric", "value", "unit", "source_id"]])
display(pd.DataFrame(reference["sources"])[["source_id", "published_on", "captured_at", "source_url", "source_file", "source_sha256"]])


## Optional explicit scenario

To calculate an exploratory scenario, supply `sportsbook_handle_usd_millions`, `sportsbook_net_margin` (a fraction, so 0.087 means 8.7%), `igaming_revenue_usd_millions`, `other_revenue_usd_millions`, and `rationale` in `scenario_inputs`. Every amount must cover the full reported US segment for the full quarter. Reconcile geography, brands, promotions, and reporting timing yourself; domestic state GGR cannot fill these inputs automatically.

Revenue = sportsbook handle × net revenue margin + iGaming revenue + other revenue. State gaming taxes are not an extra deduction from this net revenue figure. Scenarios have no approval authority.


In [ ]:
scenario = None if scenario_inputs is None else analyst_scenario(period=quarter, **scenario_inputs)
if scenario is None:
    print("No exploratory scenario supplied; no missing input was replaced with a default.")
else:
    display(pd.DataFrame([scenario["inputs"]]))
    display(pd.DataFrame([{key: value for key, value in scenario.items() if key != "inputs"}]))


## Freeze only to an explicit new destination

A snapshot records the actual write time, source identities, dataset hash, native state rows, management reference, and any explicit scenario. It never overwrites an existing directory or changes the approved model. A snapshot written after the eventual results release cannot be evaluated as a prospective expectation.


In [ ]:
if export_snapshot:
    if snapshot_destination is None or not Path(snapshot_destination).is_absolute():
        raise ValueError("Set an absolute new snapshot_destination before enabling export.")
    frozen_path = freeze_expectation(review, Path(snapshot_destination), scenario=scenario)
    print("Saved dated expectation:", frozen_path)
else:
    print("Snapshot export is disabled. This notebook has not recorded a forecast vintage.")


## Eventual result comparison

An actual must match the snapshot's exact quarter, reported-US scope, revenue metric, and USD-millions unit. It needs its original source URL, retained file/hash, publication timestamp, and capture timestamp. Unknown or pending actuals stay missing. Management-reference error and scenario error are separate; one comparison does not establish forecasting skill.


In [ ]:
if snapshot_to_evaluate is None:
    print("No frozen snapshot selected. Q3 actual comparison is pending; no error or accuracy is claimed.")
else:
    actual = None if actual_file is None else json.loads(Path(actual_file).read_text())
    display(evaluate_expectation(Path(snapshot_to_evaluate), actual, as_of=cutoff, source_root=ROOT))
assert hashlib.sha256(DB.read_bytes()).hexdigest() == before, "Read-only review changed the database."
